# 5-Fold Cross-Validation Results Analysis

This notebook analyzes the 5-fold cross-validation results for all subjects (sub-01 to sub-10) using the Specific model.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 300
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.size'] = 11
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10
plt.rcParams['legend.fontsize'] = 11
plt.rcParams['axes.linewidth'] = 1.2
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['grid.linewidth'] = 0.8

sns.set_palette("colorblind")

In [ ]:
def load_fold_metrics(subject, model, num_folds, fold):
    """Load metrics.csv for a specific fold"""
    path = Path(f"../logs/train/runs/{model}/{model}_k{num_folds}_fold{fold}_{subject}/csv/version_0/metrics.csv")
    if path.exists():
        return pd.read_csv(path)
    return None

In [ ]:
model = "nice"
num_folds = 5
subjects = [f"sub-{i:02d}" for i in range(1, 11)]

results = {}
for subject in subjects:
    fold_data = []
    for fold in range(num_folds):
        df = load_fold_metrics(subject, model, num_folds, fold)
        if df is not None:
            fold_data.append(df)
    if fold_data:
        results[subject] = fold_data
        
print(f"Loaded data for {len(results)} subjects")

In [ ]:
def extract_best_metrics(fold_data):
    """Extract best test metrics from each fold"""
    metrics = []
    for df in fold_data:
        test_row = df[df['test/loss'].notna()].iloc[-1] if not df[df['test/loss'].notna()].empty else None
        if test_row is not None:
            metrics.append({
                'test_loss': test_row['test/loss'],
                'test_top1_acc_200': test_row['test/top1_acc_retrieval_200'],
                'test_top5_acc_200': test_row['test/top5_acc_retrieval_200']
            })
    return pd.DataFrame(metrics)

## Results Summary

Calculate average test metrics across 5 folds for each subject.

In [ ]:
summary = []
for subject, fold_data in results.items():
    metrics_df = extract_best_metrics(fold_data)
    if not metrics_df.empty:
        summary.append({
            'subject': subject,
            'test_loss_mean': metrics_df['test_loss'].mean(),
            'test_loss_std': metrics_df['test_loss'].std(),
            'test_top1_acc_mean': metrics_df['test_top1_acc_200'].mean(),
            'test_top1_acc_std': metrics_df['test_top1_acc_200'].std(),
            'test_top5_acc_mean': metrics_df['test_top5_acc_200'].mean(),
            'test_top5_acc_std': metrics_df['test_top5_acc_200'].std()
        })

summary_df = pd.DataFrame(summary)
summary_df

## Comparison: FlatNet vs FlatNet+KG

Compare regular FlatNet with FlatNet enhanced by knowledge graph for each subject.

In [ ]:
def load_kg_fold_metrics(subject, model, num_folds, fold):
    """Load metrics.csv for KG version"""
    path = Path(f"../logs/train/runs/{model}/{model}_kg_k{num_folds}_fold{fold}_{subject}/csv/version_0/metrics.csv")
    if path.exists():
        return pd.read_csv(path)
    return None

kg_results = {}
for subject in subjects:
    fold_data = []
    for fold in range(num_folds):
        df = load_kg_fold_metrics(subject, model, num_folds, fold)
        if df is not None:
            fold_data.append(df)
    if fold_data:
        kg_results[subject] = fold_data
        
print(f"Loaded KG data for {len(kg_results)} subjects")

In [ ]:
kg_summary = []
for subject, fold_data in kg_results.items():
    metrics_df = extract_best_metrics(fold_data)
    if not metrics_df.empty:
        kg_summary.append({
            'subject': subject,
            'test_loss_mean': metrics_df['test_loss'].mean(),
            'test_loss_std': metrics_df['test_loss'].std(),
            'test_top1_acc_mean': metrics_df['test_top1_acc_200'].mean(),
            'test_top1_acc_std': metrics_df['test_top1_acc_200'].std(),
            'test_top5_acc_mean': metrics_df['test_top5_acc_200'].mean(),
            'test_top5_acc_std': metrics_df['test_top5_acc_200'].std()
        })

kg_summary_df = pd.DataFrame(kg_summary)
kg_summary_df

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(summary_df))
width = 0.35

colors = sns.color_palette("colorblind")
ax.bar(x - width/2, summary_df['test_top1_acc_mean'], width, yerr=summary_df['test_top1_acc_std'], 
       capsize=4, alpha=0.85, label='FlatNet', color=colors[0], edgecolor='black', linewidth=0.8)
ax.bar(x + width/2, kg_summary_df['test_top1_acc_mean'], width, yerr=kg_summary_df['test_top1_acc_std'], 
       capsize=4, alpha=0.85, label='FlatNet+KG', color=colors[1], edgecolor='black', linewidth=0.8)

ax.set_xlabel('Subject', fontweight='bold')
ax.set_ylabel('Top-1 Accuracy', fontweight='bold')
ax.set_title('Top-1 Retrieval Accuracy Comparison', fontweight='bold', pad=15)
ax.set_xticks(x)
ax.set_xticklabels(summary_df['subject'])
ax.legend(frameon=True, fancybox=False, edgecolor='black')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(summary_df))
width = 0.35

colors = sns.color_palette("colorblind")
ax.bar(x - width/2, summary_df['test_top5_acc_mean'], width, yerr=summary_df['test_top5_acc_std'], 
       capsize=4, alpha=0.85, label='FlatNet', color=colors[2], edgecolor='black', linewidth=0.8)
ax.bar(x + width/2, kg_summary_df['test_top5_acc_mean'], width, yerr=kg_summary_df['test_top5_acc_std'], 
       capsize=4, alpha=0.85, label='FlatNet+KG', color=colors[3], edgecolor='black', linewidth=0.8)

ax.set_xlabel('Subject', fontweight='bold')
ax.set_ylabel('Top-5 Accuracy', fontweight='bold')
ax.set_title('Top-5 Retrieval Accuracy Comparison', fontweight='bold', pad=15)
ax.set_xticks(x)
ax.set_xticklabels(summary_df['subject'])
ax.legend(frameon=True, fancybox=False, edgecolor='black')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(summary_df))
width = 0.35

colors = sns.color_palette("colorblind")
ax.bar(x - width/2, summary_df['test_loss_mean'], width, yerr=summary_df['test_loss_std'], 
       capsize=4, alpha=0.85, label='FlatNet', color=colors[4], edgecolor='black', linewidth=0.8)
ax.bar(x + width/2, kg_summary_df['test_loss_mean'], width, yerr=kg_summary_df['test_loss_std'], 
       capsize=4, alpha=0.85, label='FlatNet+KG', color=colors[5], edgecolor='black', linewidth=0.8)

ax.set_xlabel('Subject', fontweight='bold')
ax.set_ylabel('Test Loss', fontweight='bold')
ax.set_title('Test Loss Comparison', fontweight='bold', pad=15)
ax.set_xticks(x)
ax.set_xticklabels(summary_df['subject'])
ax.legend(frameon=True, fancybox=False, edgecolor='black')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
improvement = pd.DataFrame({
    'subject': summary_df['subject'],
    'top1_improvement': ((kg_summary_df['test_top1_acc_mean'].values - summary_df['test_top1_acc_mean'].values) / summary_df['test_top1_acc_mean'].values * 100),
    'top5_improvement': ((kg_summary_df['test_top5_acc_mean'].values - summary_df['test_top5_acc_mean'].values) / summary_df['test_top5_acc_mean'].values * 100),
    'loss_improvement': ((summary_df['test_loss_mean'].values - kg_summary_df['test_loss_mean'].values) / summary_df['test_loss_mean'].values * 100)
})

print("Improvement with Knowledge Graph (%):")
print(f"Average Top-1 Accuracy: {improvement['top1_improvement'].mean():.2f}%")
print(f"Average Top-5 Accuracy: {improvement['top5_improvement'].mean():.2f}%")
print(f"Average Loss Reduction: {improvement['loss_improvement'].mean():.2f}%")
print("\nPer-subject improvement:")
improvement